# **Life Expectancy and GDP Analysis**

## **1. Introduction:**

This project examines the relationship between gross domestic product (GDP) and life expectancy across six countries from 2000 to 2015. Using Python and pandas, the analysis explores changes over time, compares country-level trends, and investigates whether higher GDP is associated with higher life expectancy in the selected dataset.

## **2. Problem Statement**

Economic growth and population health are important indicators of a country's development. However, changes in GDP do not necessarily correspond directly to improvements in life expectancy.

This project investigates how GDP and life expectancy evolved across Chile, China, Germany, Mexico, the United States of America, and Zimbabwe between 2000 and 2015.

### Objectives

1. Examine changes in life expectancy across the six countries.
2. Compare GDP trends over the study period.
3. Investigate the relationship between GDP and life expectancy.
4. Identify differences in country-level trends and improvements.
5. Communicate findings while recognizing the limitations of the dataset.

## **3. Dataset Source and Attribution**

The CSV dataset was obtained from Codecademy as part of its learning materials. The dataset is used here for educational and portfolio purposes. 

### Variables

| Column | Description |
|---|---|
| Country | Name of the country |
| Year | Year of observation |
| Life expectancy at birth (years) | Life expectancy measured in years |
| GDP | Gross Domestic Product, subject to the units and definition provided by the dataset source |

Each country has one observation per year. The analysis will verify this structure before proceeding.

This data is based on World Health Organization and the World Bank so GDP values are reported in current U.S. dollars and are not adjusted for inflation. As a result, observed GDP growth reflects both economic expansion and changes in price levels over time.

### Scope and Limitations

The dataset contains only six countries and does not establish whether GDP changes cause changes in life expectancy. Findings will be interpreted as patterns within the observed countries and period.

## **4. Data Loading, Inspection and Assessment**
This dataset originates from Codecademy's Life Expectancy and GDP portfolio project. The file has been renamed to maintain a consistent project naming convention.

In [1]:
# Import pandas for working with tabular data.
import pandas as pd

# Import Path to construct a file path in a platform-independent way.
from pathlib import Path

# Define the path to the raw life expectancy and GDP data file.
data_path = Path("../data/raw/life_expectancy_gdp.csv")

# Load the original dataset
df_raw = pd.read_csv(data_path)

# Create a working copy for cleaning and preparation
df = df_raw.copy()

#### **a) Preview of data frame**
The first few rows of the dataset are displayed to verify that the data loaded correctly and to gain an initial understanding of its structure and contents.

In [2]:
# Display the first 20 rows of the DataFrame to preview the data.
df.head(20)

,Country,Year,Life expectancy at birth (years),GDP
0,Chile,2000,77.3,7.786093e+10
1,Chile,2001,77.3,7.097992e+10
2,Chile,2002,77.8,6.973681e+10
3,Chile,2003,77.9,7.564346e+10
4,Chile,2004,78.0,9.921039e+10
5,Chile,2005,78.4,1.229650e+11
6,Chile,2006,78.9,1.547880e+11
7,Chile,2007,78.9,1.736060e+11
8,Chile,2008,79.6,1.796380e+11
9,Chile,2009,79.3,1.723890e+11


Observations: 
- The dataset contains the four variables identified in the previous section: Country, Year, Life Expectancy at Birth (Years), and GDP.
- GDP values are displayed in scientific notation due to their large magnitude, which reduces readability during inspection and analysis.
- The column name "Life expectancy at birth (years)" is lengthy and can be shortened to improve readability and ease of use in code.

#### **b) DataFrame Summary**
The DataFrame summary is reviewed to verify the dataset's dimensions, data types, and completeness before further analysis.

In [3]:
# Display a concise summary of the DataFrame, including its columns,
# data types, non-null counts, and memory usage.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 4 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Country                           96 non-null     str    
 1   Year                              96 non-null     int64  
 2   Life expectancy at birth (years)  96 non-null     float64
 3   GDP                               96 non-null     float64
dtypes: float64(2), int64(1), str(1)
memory usage: 3.1 KB


Observations: All columns seemed to have correct data types. No missing values and 96 observations for all years spread across all mentioned countries.

#### **c) Quality check**
The dataset is checked to verify that all expected countries are present and that each country contains observations for every year from 2000 to 2015. These checks help confirm the completeness of the data before proceeding with analysis

In [4]:
df["Country"].unique()

<StringArray>
['Chile', 'China', 'Germany', 'Mexico', 'United States of America',
 'Zimbabwe']
Length: 6, dtype: str

This code keeps only countries whose missing-year list is non-empty. If the final result is empty, every country has at least one row for every year from 2000 to 2015.

Note: this checks that a country has each year at least once; it doesn’t check for duplicate country-year rows.

In [5]:
# Create a set of all years that each country is expected to have (2000–2015).
expected_years = set(range(2000, 2016))

# For each country, find and sort the expected years that are not present.
missing_years = df.groupby("Country")["Year"].apply(
    lambda years: sorted(expected_years - set(years.dropna()))
)

# Keep only countries that are missing one or more years.
missing_years = missing_years[missing_years.apply(bool)]

print("Expected years present for all countries:",
      missing_years.empty)


Expected years present for all countries: True


## **5.  Data Preparation**
Before analysis, the dataset is reviewed for duplicate records and prepared for readability and consistency. Variable names are standardized, data types are verified, and GDP values are converted to billions of U.S. dollars to improve interpretability in tables and visualizations.

The dataset is examined for duplicate rows and duplicate country-year combinations to ensure that each observation is unique.

In [6]:
print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate country-year pairs:",
    df.duplicated(subset=["Country", "Year"]).sum()
)

Duplicate rows: 0
Duplicate country-year pairs: 0


No duplicate rows were identified.
No duplicate country-year combinations were found.
Each country-year pair appears exactly once in the dataset.

The dataset is prepared for analysis by renaming variables, validating data types, and converting GDP values to billions for easier interpretation.

In [7]:
# Rename the life expectancy column to a shorter, easier-to-use name.
df = df.rename(columns={
    "Life expectancy at birth (years)": "Life_Expectancy_Years"
})

# Convert both columns to numeric values.
# errors="raise" reports an error if a value cannot be converted.
df["Life_Expectancy_Years"] = pd.to_numeric(
    df["Life_Expectancy_Years"], errors="raise"
)

# Round life expectancy values to one decimal place.
df["Life_Expectancy_Years"] = df["Life_Expectancy_Years"].round(1)

# Convert GDP to billions and replace the original GDP column.
df["GDP"] = df["GDP"] / 1_000_000_000

# Rename the column to reflect the new unit.
df = df.rename(columns={"GDP": "GDP_Billions"})

In [8]:
# Preview of dataFrame after cleaning and formatting.
df.head(25)

,Country,Year,Life_Expectancy_Years,GDP_Billions
0,Chile,2000,77.3,77.860932
1,Chile,2001,77.3,70.979924
2,Chile,2002,77.8,69.736811
3,Chile,2003,77.9,75.643460
4,Chile,2004,78.0,99.210393
5,Chile,2005,78.4,122.965000
6,Chile,2006,78.9,154.788000
7,Chile,2007,78.9,173.606000
8,Chile,2008,79.6,179.638000
9,Chile,2009,79.3,172.389000


- The life expectancy column was renamed to improve readability and consistency.
- Life expectancy values were converted to numeric format and rounded to one decimal place.
- GDP values were converted from U.S. dollars to billions of U.S. dollars.
- The GDP column was renamed to reflect its new unit of measurement.
- The resulting dataset is easier to interpret and visualize while preserving the original information.

The cleaned dataset is saved as a CSV file for use in the exploratory analysis.

In [9]:
# Define the output path
cleaned_data_path = Path("../data/cleaned/life_expectancy_gdp_cleaned.csv")

# Create the directory if it doesn't exist
cleaned_data_path.parent.mkdir(parents=True, exist_ok=True)

# Save the prepared DataFrame to CSV
df.to_csv(cleaned_data_path, index=False)

print(f"Cleaned data saved to: {cleaned_data_path}")

Cleaned data saved to: ..\data\cleaned\life_expectancy_gdp_cleaned.csv


## **6. Preparation Summary** 
The dataset was inspected for completeness, consistency, and duplicate records before analysis. Column names were standardized for readability, and GDP values were converted to billions of U.S. dollars to improve interpretability.

No missing values, duplicate records, or missing country-year combinations were identified. The cleaned dataset was saved for use in the exploratory analysis of GDP and life expectancy trends.